# this notebook is to combine the data multiple repetition...

In [1]:
import numpy as np 
import pandas as pd 
import cv2 
import sys 
import os 
import glob 

sys.path.append('../../')
sys.path.append('../../event_tools')
import event_tools.tools as tl 
import event_tools.e2v  as e2v 



In [3]:
individual_artificial_directory = "./artificial_data/individual"
artificial_directory = "./artificial_data/same_combined"

data_class = "class2"
# filename =  "user02_fluorescent" 
# filename =  "user02_fluorescent_led"
# filename =  "user02_lab"  
# filename =  "user02_led"  
filename =  "user02_natural" 

os.makedirs(os.path.join(artificial_directory,data_class), exist_ok=True)

In [62]:
import re 
# Short forms for lighting conditions
lighting_map = {
    'flourescent_led': 'fl',
    'fluorescent': 'f',
    'lab': 'la',
    'led': 'le',
    'natural': 'n'
}

def extract_info(filename):
    lighting_part = filename.split('_')[-1]
    lighting_type, count = lighting_part.split('-')
    count = int(count.replace('.csv', ''))
    return lighting_type, count

def combine_csvs(filepaths):
    dfs = []
    total_count = 0
    lighting_parts = []

    for filepath in filepaths:
        lighting_type, count = extract_info(os.path.basename(filepath))
        short = lighting_map[lighting_type]
        lighting_parts.append(short)
        total_count += count
        df = pd.read_csv(filepath)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df, '+'.join(lighting_parts), total_count


class_folders = sorted(glob.glob(individual_artificial_directory + "/*"))
class_info = "class[0-9]"

# class_folders[0]
# os.path.basename(glob.glob(class_folders[0] + "/*")[0].split("_")[-1])
a, lighting_parts, total_count = combine_csvs(glob.glob(class_folders[0] + "/*")[:2])
b = pd.read_csv(glob.glob(class_folders[0] + "/*")[0])
c = pd.read_csv(glob.glob(class_folders[0] + "/*")[1])

In [61]:
len(a[0])

3

In [10]:
events = tl.load_event_data(
    # './event_csv/split_data/artificial/a_b7_a.csv') # Replace with your actual file path
    # '../../event_csv/split_data/class5/user02_lab.csv')  # Replace with your actual file path
    f"../../event_csv/split_data/{data_class}/{filename}.csv")  # Replace with your actual file path
# save_event_as_video(events, 'test.mp4', frame_per_second=1/30)

In [27]:
height, width = 128, 128 # event camera resolution 
# create image 
image = np.zeros((height, width, 3), dtype=np.uint16)

# if event_points:
#     events = events[:event_points]

image_scale = 1
frame_per_second = 1/120 # frame per second 
prev_x, prev_y, prev_time = 0, 0, 0 
peak, spike = 255, 255
decay = 0.005

test_data_frame = [] 


for event_idx, event in enumerate(events):
    time_stamp, x, y = event[0], int(event[1]), int(event[2]) 
    # print(time_stamp, x, y)
    test_data_frame.append([time_stamp, x, y])
    
    # spike of the neuron 
    image[y, x] = spike 
    
    # decay of the neuron
    image = image - (decay * image ) #* (time_stamp - prev_time))
    image[image < 0] = 0 
    
    # print(event_idx, image.max())
    image_display = image.astype(np.uint8)
    
    cv2.imshow('test', cv2.resize(image_display, (width * image_scale, height * image_scale)))
    
    if ((time_stamp - prev_time) >  (frame_per_second * 100000)): 
        prev_time = time_stamp 
    
        key = cv2.waitKey(1)
        if key == ord('q'):
            break
    ...
cv2.destroyAllWindows()

In [28]:
segmented_events = test_data_frame
a = pd.DataFrame(segmented_events, columns=['timestamp', 'x', 'y'])

In [30]:
height, width = 128, 128 # event camera resolution 
# create image 
image = np.zeros((height, width, 3), dtype=np.uint16)

# if event_points:
#     events = events[:event_points]

image_scale = 1
frame_per_second = 1/120 # frame per second 
prev_x, prev_y, prev_time = 0, 0, 0 
peak, spike = 255, 255
decay = 0.005

# test_data_frame = [] 


for event_idx, event in enumerate(test_data_frame):
    time_stamp, x, y = event[0], int(event[1]), int(event[2]) 
    # print(time_stamp, x, y)
    # test_data_frame.append([time_stamp, x, y])
    
    # spike of the neuron 
    image[y, x] = spike 
    
    # decay of the neuron
    image = image - (decay * image ) #* (time_stamp - prev_time))
    image[image < 0] = 0 
    
    # print(event_idx, image.max())
    image_display = image.astype(np.uint8)
    
    cv2.imshow('test', cv2.resize(image_display, (width * image_scale, height * image_scale)))
    
    if ((time_stamp - prev_time) >  (frame_per_second * 200000)): 
        prev_time = time_stamp 
    
        key = cv2.waitKey(1)
        if key == ord('q'):
            break
    ...
cv2.destroyAllWindows()

In [31]:
count_input = input("Enter the number of events to count: ")

In [32]:
a.to_csv(f"./{artificial_directory}/{data_class}/{filename}-{count_input}.csv", index=False)

In [33]:
a

,timestamp,x,y
0,38707747.0,45,61
1,38707756.0,41,60
2,38707774.0,47,56
3,38707842.0,44,55
4,38708004.0,39,55
...,...,...,...
177395,43354746.0,26,85
177396,43354825.0,36,93
177397,43354832.0,28,87
177398,43354842.0,23,93


In [64]:
ls

a_3.csv                       segment_data.ipynb
artifical_data_compose.ipynb  test_count.ipynb
artificial_data/              test.csv
csv_clean_to_txt.ipynb        test_ipca.ipynb
csv_to_video.ipynb            test_mae_obo.ipynb
cycle_count.ipynb             test_main.ipynb
data_compress.ipynb           test_mean.ipynb
data_vis_analysis.ipynb       test_mk.ipynb
export_script_pca_x_y.py      test_seg.csv
figs/                         test_without_pca_and_replace_with_EMA.ipynb
main.ipynb                    test_without_pca.ipynb
my_plot.png                   utils.ipynb
package.ipynb                 visualization.ipynb
segment_data_combined.ipynb


In [66]:
import os
import pandas as pd
from itertools import combinations
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Short forms for lighting conditions
lighting_map = {
    'flourescent_led': 'fl',
    'fluorescent': 'f',
    'lab': 'la',
    'led': 'le',
    'natural': 'n'
}

BASE_DIR = "./artificial_data/"

INPUT_DIR = os.path.join(BASE_DIR, 'individual')
OUTPUT_DIR_SAME = os.path.join(BASE_DIR, 'combined_same_class')
OUTPUT_DIR_MULTI = os.path.join(BASE_DIR, 'combined_multi_class')

def extract_info(filename):
    lighting_part = filename.split('_')[1]
    lighting_type, count = lighting_part.split('-')
    count = int(count.replace('.csv', ''))
    return lighting_type, count

def combine_csvs(filepaths):
    dfs = []
    total_count = 0
    lighting_parts = []

    for filepath in filepaths:
        lighting_type, count = extract_info(os.path.basename(filepath))
        short = lighting_map[lighting_type]
        lighting_parts.append(short)
        total_count += count
        df = pd.read_csv(filepath)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df, '+'.join(lighting_parts), total_count

def save_combined_file(df, filename, output_dir):
    out_path = os.path.join(output_dir, filename)
    df.to_csv(out_path, index=False)
    print(f"[+] Merged and saved: {out_path}")

def process_same_class_combo(combo):
    df, label_part, total_count = combine_csvs(combo)
    user_id = combo[0].split('_')[0]
    filename = f"{user_id}+{label_part}-{total_count}.csv"
    save_combined_file(df, filename, OUTPUT_DIR_SAME)

def process_multi_class_combo(combo_classes, file1, file2):
    df, label_part, total_count = combine_csvs([file1, file2])
    user_id = file1.split('_')[0]
    class_names = ','.join(sorted([combo_classes[0], combo_classes[1]]))
    filename = f"c_{class_names}_{user_id}+{label_part}-{total_count}.csv"
    save_combined_file(df, filename, OUTPUT_DIR_MULTI)

def process_same_class():
    print("Processing same-class merges...\n")
    with ThreadPoolExecutor() as executor:
        for class_dir in os.listdir(INPUT_DIR):
            class_path = os.path.join(INPUT_DIR, class_dir)
            if not os.path.isdir(class_path):
                continue

            files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.csv')]
            combos = list(combinations(files, 2))
            executor.map(process_same_class_combo, combos)

def process_multi_class():
    print("\nProcessing multi-class merges...\n")
    class_dirs = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))]

    with ThreadPoolExecutor() as executor:
        tasks = []
        for combo_classes in combinations(class_dirs, 2):
            class1_files = [os.path.join(INPUT_DIR, combo_classes[0], f)
                            for f in os.listdir(os.path.join(INPUT_DIR, combo_classes[0])) if f.endswith('.csv')]
            class2_files = [os.path.join(INPUT_DIR, combo_classes[1], f)
                            for f in os.listdir(os.path.join(INPUT_DIR, combo_classes[1])) if f.endswith('.csv')]

            for file1 in class1_files:
                for file2 in class2_files:
                    tasks.append((combo_classes, file1, file2))

        # Map using unpacked args
        executor.map(lambda args: process_multi_class_combo(*args), tasks)

if __name__ == "__main__":
    process_same_class()
    process_multi_class()

Processing same-class merges...

[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+n-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+le-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+le-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+le-10.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+f-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+f-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+n-13.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+le-13.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+la-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+le+n-11.csv
[+] Merged and saved: ./artificial_data/combined_same_class/./artificial+n+f-10.csv
[+] Merged and saved: ./artificial_data/com

In [6]:
import os
import traceback
import pandas as pd
from itertools import combinations
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Short forms for lighting conditions
lighting_map = {
    'fluorescent_led': 'fl',
    'fluorescent': 'f',
    'lab': 'la',
    'led': 'le',
    'natural': 'n'
}

BASE_DIR = "./artificial_data/"

INPUT_DIR = os.path.join(BASE_DIR, 'individual')
OUTPUT_DIR_SAME = os.path.join(BASE_DIR, 'combined_same_class')
OUTPUT_DIR_MULTI = os.path.join(BASE_DIR, 'combined_multi_class')

Path(OUTPUT_DIR_SAME).mkdir(exist_ok=True)
Path(OUTPUT_DIR_MULTI).mkdir(exist_ok=True)


def extract_info(filename):
    # filename: user02_fluorescent_led-3.csv
    base = os.path.basename(filename).replace('.csv', '')  # remove extension
    # parts = base.split('_', 1)  # split once to preserve underscores in lighting
    parts = base.replace('user02_', '')
    if len(parts) < 2:
        raise ValueError(f"Filename not formatted correctly: {filename}")
    
    lighting_and_count = parts # [1]  # e.g. fluorescent_led-3
    # Now split from the right to safely get count even with underscores
    lighting_type, count_str = lighting_and_count.rsplit('-', 1)
    return lighting_type, int(count_str)


def combine_csvs(filepaths):
    dfs = []
    total_count = 0
    lighting_parts = []

    for filepath in filepaths:
        lighting_type, count = extract_info(os.path.basename(filepath))
        short = lighting_map[lighting_type]
        lighting_parts.append(short)
        total_count += count
        df = pd.read_csv(filepath)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df, '+'.join(lighting_parts), total_count

def save_combined_file(df, filename, output_dir):
    out_path = os.path.join(output_dir, filename)
    df.to_csv(out_path, index=False)
    print(f"[+] Merged and saved: {out_path}")

def process_same_class_combo(combo):
    try:
        df, label_part, total_count = combine_csvs(combo)
        user_id = os.path.basename(combo[0]).split('_')[0]  # Fixed
        filename = f"{user_id}+{label_part}-{total_count}.csv"
        save_combined_file(df, filename, OUTPUT_DIR_SAME)
    except Exception as e:
        print(f"[!] Error in same-class merge: {combo}: {e}")
        traceback.print_exc()

def process_multi_class_combo(combo_classes, file1, file2):
    try:
        df, label_part, total_count = combine_csvs([file1, file2])
        user_id = os.path.basename(file1).split('_')[0]  # Fixed
        class_names = ','.join(sorted([combo_classes[0], combo_classes[1]]))
        filename = f"c_{class_names}_{user_id}+{label_part}-{total_count}.csv"
        save_combined_file(df, filename, OUTPUT_DIR_MULTI)
    except Exception as e:
        print(f"[!] Error merging {file1} and {file2}: {e}")

def process_same_class():
    print("🔁 Processing same-class merges...\n")
    with ThreadPoolExecutor() as executor:
        for class_dir in os.listdir(INPUT_DIR):
            class_path = os.path.join(INPUT_DIR, class_dir)
            if not os.path.isdir(class_path):
                continue

            files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.csv')]
            combos = list(combinations(files, 2))
            if combos:
                print(f"  → {class_dir}: {len(combos)} combinations")
            executor.map(process_same_class_combo, combos)

def process_multi_class():
    print("\n🔁 Processing multi-class merges...\n")
    class_dirs = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))]

    with ThreadPoolExecutor() as executor:
        tasks = []
        for combo_classes in combinations(class_dirs, 2):
            print(f"  → Class combo: {combo_classes}")
            class1_path = os.path.join(INPUT_DIR, combo_classes[0])
            class2_path = os.path.join(INPUT_DIR, combo_classes[1])

            class1_files = [os.path.join(class1_path, f) for f in os.listdir(class1_path) if f.endswith('.csv')]
            class2_files = [os.path.join(class2_path, f) for f in os.listdir(class2_path) if f.endswith('.csv')]

            print(f"     Class1 files: {len(class1_files)}, Class2 files: {len(class2_files)}")

            for file1 in class1_files:
                for file2 in class2_files:
                    tasks.append((combo_classes, file1, file2))

        print(f"[~] Preparing to process {len(tasks)} multi-class merge combinations.\n")
        executor.map(lambda args: process_multi_class_combo(*args), tasks)

if __name__ == "__main__":
    process_same_class()
    process_multi_class()

🔁 Processing same-class merges...

  → class3: 105 combinations
  → class4: 105 combinations
  → class2: 105 combinations
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+n-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+fl-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-10.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+le-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+n-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+f-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+n-13.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+fl-10.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+f-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-13.csv
[+] Merged and sav

KeyboardInterrupt: 

[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+n-11.csv


[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+fl-11.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+le+la-11.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+fl-8.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+la-6.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+le-8.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+la-8.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+f-8.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+fl-11.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+le-11.csv
[+] Merged and saved: ./artificial_data/combined_multi_class/c_class3,class4_user02+f+n-13.csv
[+] Merged and saved: ./artificial_data/combin

In [ ]:
import os
import pandas as pd
from itertools import combinations, product
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Short forms for lighting conditions
lighting_map = {
    'fluorescent_led': 'fl',
    'fluorescent': 'f',
    'lab': 'la',
    'led': 'le',
    'natural': 'n'
}

BASE_DIR = "./artificial_data/"

INPUT_DIR = os.path.join(BASE_DIR, 'individual')
OUTPUT_DIR_SAME = os.path.join(BASE_DIR, 'combined_same_class')
OUTPUT_DIR_MULTI = os.path.join(BASE_DIR, 'combined_multi_class')

MERGE_COMBO_SIZE = 2  # Change this to the desired size of combinations

Path(OUTPUT_DIR_SAME).mkdir(exist_ok=True)
Path(OUTPUT_DIR_MULTI).mkdir(exist_ok=True)

# ===== Helpers =====
def extract_info(filename):
    base = os.path.basename(filename).replace('.csv', '')
    parts = base.split('_', 1)
    if len(parts) < 2:
        raise ValueError(f"Filename format error: {filename}")
    lighting_and_count = parts[1]
    lighting_type, count_str = lighting_and_count.rsplit('-', 1)
    return lighting_type, int(count_str)

def combine_csvs(filepaths):
    dfs = []
    total_count = 0
    lighting_parts = []

    for filepath in filepaths:
        lighting_type, count = extract_info(filepath)
        short = lighting_map.get(lighting_type, lighting_type[:2])
        lighting_parts.append(short)
        total_count += count
        df = pd.read_csv(filepath)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df, '+'.join(lighting_parts), total_count

def save_combined_file(df, filename, output_dir):
    out_path = os.path.join(output_dir, filename)
    df.to_csv(out_path, index=False)
    print(f"[+] Merged and saved: {out_path}")

# ===== Same-Class Merging =====
def process_same_class_combo(file_list):
    try:
        df, label_part, total_count = combine_csvs(file_list)
        user_id = os.path.basename(file_list[0]).split('_')[0]
        filename = f"{user_id}+{label_part}-{total_count}.csv"
        save_combined_file(df, filename, OUTPUT_DIR_SAME)
    except Exception as e:
        print(f"[!] Error in same-class merge: {[os.path.basename(f) for f in file_list]}: {e}")

def process_same_class():
    print(f"🔁 Processing same-class merges (merge size = {MERGE_COMBO_SIZE})...\n")
    with ThreadPoolExecutor() as executor:
        for class_dir in os.listdir(INPUT_DIR):
            class_path = os.path.join(INPUT_DIR, class_dir)
            if not os.path.isdir(class_path):
                continue

            files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.csv')]
            if len(files) < MERGE_COMBO_SIZE:
                continue

            combos = list(combinations(files, MERGE_COMBO_SIZE))
            print(f"  → {class_dir}: {len(combos)} combinations")
            executor.map(process_same_class_combo, combos)

# ===== Multi-Class Merging =====
def process_multi_class_combo(class_group, file_list):
    try:
        df, _, _ = combine_csvs(file_list)
        user_id = os.path.basename(file_list[0]).split('_')[0]

        filename_parts = []
        for class_name, file in zip(class_group, file_list):
            lighting_type, count = extract_info(file)
            short = lighting_map.get(lighting_type, lighting_type[:2])
            filename_parts.append(f"{class_name}:{short}-{count}")

        combo_name = '+'.join(filename_parts)
        filename = f"c_{combo_name}_{user_id}.csv"
        save_combined_file(df, filename, OUTPUT_DIR_MULTI)
    except Exception as e:
        print(f"[!] Error in multi-class merge: {[os.path.basename(f) for f in file_list]}: {e}")

def process_multi_class():
    print(f"\n🔁 Processing multi-class merges (combo size = {MERGE_COMBO_SIZE})...\n")
    class_dirs = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))]
    if len(class_dirs) < MERGE_COMBO_SIZE:
        print("[!] Not enough classes to process multi-class combos.")
        return

    with ThreadPoolExecutor() as executor:
        tasks = []

        for class_group in combinations(class_dirs, MERGE_COMBO_SIZE):
            print(f"  → Class group: {class_group}")
            all_class_files = []

            for class_name in class_group:
                class_path = os.path.join(INPUT_DIR, class_name)
                class_files = [os.path.join(class_path, f)
                               for f in os.listdir(class_path) if f.endswith('.csv')]
                if not class_files:
                    print(f"    [!] Skipping {class_name} (no .csv files)")
                    break
                all_class_files.append(class_files)
            else:
                for file_combo in product(*all_class_files):  # Cartesian product
                    tasks.append((class_group, list(file_combo)))

        print(f"[~] Preparing to process {len(tasks)} multi-class merge combinations.\n")
        executor.map(lambda args: process_multi_class_combo(*args), tasks)

# ===== Run =====
if __name__ == "__main__":
    process_same_class()
    process_multi_class()


🔁 Processing same-class merges (merge size = 2)...

  → class3: 105 combinations
  → class4: 105 combinations
  → class2: 105 combinations
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+n-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+le-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-10.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+le-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+fl-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+n-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+la-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+fl-10.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+n-13.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+f-8.csv
[

In [10]:
def combine_csvs(filepaths):
    dfs = []
    total_count = 0
    lighting_parts = []
    last_timestamp = None

    for i, filepath in enumerate(filepaths):
        lighting_type, count = extract_info(filepath)
        short = lighting_map.get(lighting_type, lighting_type[:2])
        lighting_parts.append(short)
        total_count += count

        df = pd.read_csv(filepath)

        if 'timestamp' not in df.columns:
            raise ValueError(f"'timestamp' column missing in {filepath}")

        # Optional: add a source column (e.g., "user02+f-3.csv")
        df['source_file'] = os.path.basename(filepath)

        # Adjust timestamps for continuity
        if i > 0:
            original_start = df['timestamp'].iloc[0]
            time_diff = df['timestamp'].diff().median()
            if pd.isna(time_diff) or time_diff <= 0:
                time_diff = 1  # fallback in case timestamps are flat or corrupted

            shift_value = last_timestamp + time_diff - original_start
            df['timestamp'] = df['timestamp'] + shift_value

        last_timestamp = df['timestamp'].iloc[-1]
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df, '+'.join(lighting_parts), total_count

In [11]:
import os
import pandas as pd
from itertools import combinations, product
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Short forms for lighting conditions
lighting_map = {
    'fluorescent_led': 'fl',
    'fluorescent': 'f',
    'lab': 'la',
    'led': 'le',
    'natural': 'n'
}

BASE_DIR = "./artificial_data/"

INPUT_DIR = os.path.join(BASE_DIR, 'individual')
OUTPUT_DIR_SAME = os.path.join(BASE_DIR, 'combined_same_class')
OUTPUT_DIR_MULTI = os.path.join(BASE_DIR, 'combined_multi_class')

MERGE_COMBO_SIZE = 2  # Change this to the desired size of combinations

Path(OUTPUT_DIR_SAME).mkdir(exist_ok=True)
Path(OUTPUT_DIR_MULTI).mkdir(exist_ok=True)

# ===== Helpers =====
def extract_info(filename):
    base = os.path.basename(filename).replace('.csv', '')
    parts = base.split('_', 1)
    if len(parts) < 2:
        raise ValueError(f"Filename format error: {filename}")
    lighting_and_count = parts[1]
    lighting_type, count_str = lighting_and_count.rsplit('-', 1)
    return lighting_type, int(count_str)

def combine_csvs(filepaths):
    dfs = []
    total_count = 0
    lighting_parts = []
    last_timestamp = None

    for i, filepath in enumerate(filepaths):
        lighting_type, count = extract_info(filepath)
        short = lighting_map.get(lighting_type, lighting_type[:2])
        lighting_parts.append(short)
        total_count += count

        df = pd.read_csv(filepath)

        if 'timestamp' not in df.columns:
            raise ValueError(f"'timestamp' column missing in {filepath}")

        # Optional: add a source column (e.g., "user02+f-3.csv")
        df['source_file'] = os.path.basename(filepath)

        # Adjust timestamps for continuity
        if i > 0:
            original_start = df['timestamp'].iloc[0]
            time_diff = df['timestamp'].diff().median()
            if pd.isna(time_diff) or time_diff <= 0:
                time_diff = 1  # fallback in case timestamps are flat or corrupted

            shift_value = last_timestamp + time_diff - original_start
            df['timestamp'] = df['timestamp'] + shift_value

        last_timestamp = df['timestamp'].iloc[-1]
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    return combined_df, '+'.join(lighting_parts), total_count

def save_combined_file(df, filename, output_dir):
    out_path = os.path.join(output_dir, filename)
    df.to_csv(out_path, index=False)
    print(f"[+] Merged and saved: {out_path}")

# ===== Same-Class Merging =====
def process_same_class_combo(file_list):
    try:
        df, label_part, total_count = combine_csvs(file_list)
        user_id = os.path.basename(file_list[0]).split('_')[0]
        filename = f"{user_id}+{label_part}-{total_count}.csv"
        save_combined_file(df, filename, OUTPUT_DIR_SAME)
    except Exception as e:
        print(f"[!] Error in same-class merge: {[os.path.basename(f) for f in file_list]}: {e}")

def process_same_class():
    print(f"🔁 Processing same-class merges (merge size = {MERGE_COMBO_SIZE})...\n")
    with ThreadPoolExecutor() as executor:
        for class_dir in os.listdir(INPUT_DIR):
            class_path = os.path.join(INPUT_DIR, class_dir)
            if not os.path.isdir(class_path):
                continue

            files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith('.csv')]
            if len(files) < MERGE_COMBO_SIZE:
                continue

            combos = list(combinations(files, MERGE_COMBO_SIZE))
            print(f"  → {class_dir}: {len(combos)} combinations")
            executor.map(process_same_class_combo, combos)

# ===== Multi-Class Merging =====
def process_multi_class_combo(class_group, file_list):
    try:
        df, _, _ = combine_csvs(file_list)
        user_id = os.path.basename(file_list[0]).split('_')[0]

        filename_parts = []
        for class_name, file in zip(class_group, file_list):
            lighting_type, count = extract_info(file)
            short = lighting_map.get(lighting_type, lighting_type[:2])
            filename_parts.append(f"{class_name}:{short}-{count}")

        combo_name = '+'.join(filename_parts)
        filename = f"c_{combo_name}_{user_id}.csv"
        save_combined_file(df, filename, OUTPUT_DIR_MULTI)
    except Exception as e:
        print(f"[!] Error in multi-class merge: {[os.path.basename(f) for f in file_list]}: {e}")

def process_multi_class():
    print(f"\n🔁 Processing multi-class merges (combo size = {MERGE_COMBO_SIZE})...\n")
    class_dirs = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))]
    if len(class_dirs) < MERGE_COMBO_SIZE:
        print("[!] Not enough classes to process multi-class combos.")
        return

    with ThreadPoolExecutor() as executor:
        tasks = []

        for class_group in combinations(class_dirs, MERGE_COMBO_SIZE):
            print(f"  → Class group: {class_group}")
            all_class_files = []

            for class_name in class_group:
                class_path = os.path.join(INPUT_DIR, class_name)
                class_files = [os.path.join(class_path, f)
                               for f in os.listdir(class_path) if f.endswith('.csv')]
                if not class_files:
                    print(f"    [!] Skipping {class_name} (no .csv files)")
                    break
                all_class_files.append(class_files)
            else:
                for file_combo in product(*all_class_files):  # Cartesian product
                    tasks.append((class_group, list(file_combo)))

        print(f"[~] Preparing to process {len(tasks)} multi-class merge combinations.\n")
        executor.map(lambda args: process_multi_class_combo(*args), tasks)

# ===== Run =====
if __name__ == "__main__":
    process_same_class()
    process_multi_class()


🔁 Processing same-class merges (merge size = 2)...

  → class3: 105 combinations
  → class4: 105 combinations
  → class2: 105 combinations
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+n-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-10.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+le-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+fl-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+n-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+le-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+le-13.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+fl+f-6.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+f-8.csv
[+] Merged and saved: ./artificial_data/combined_same_class/user02+n+n-13.csv
[

parsing the files 

In [12]:
import os
import re
import json

# Short forms for lighting conditions
lighting_map = {
    'fluorescent_led': 'fl',
    'fluorescent': 'f',
    'lab': 'la',
    'led': 'le',
    'natural': 'n'
}

BASE_DIR = "./artificial_data/"

# ===== CONFIG =====
SAME_DIR = os.path.join(BASE_DIR, 'combined_same_class')
MULTI_DIR = os.path.join(BASE_DIR,'combined_multi_class')
SAVE_JSON = True  # Set to False to only print

# ===== Helpers =====

def parse_same_class_filename(filename):
    base = filename.replace('.csv', '')
    match = re.match(r'(?P<user_id>user\d+)\+(?P<lighting>.+)-(?P<count>\d+)', base)
    if not match:
        raise ValueError(f"Invalid same-class filename: {filename}")
    
    lighting_conditions = match.group("lighting").split("+")
    return {
        'filename': filename,
        'user_id': match.group("user_id"),
        'lighting_conditions': lighting_conditions,
        'total_count': int(match.group("count"))
    }

def parse_multi_class_filename(filename):
    base = filename.replace('.csv', '')
    try:
        parts = base.split("_")
        user_id = parts[-1]
        combo_part = "_".join(parts[1:-1])
        combos = combo_part.split('+')
        
        entries = []
        for combo in combos:
            class_part, light_count = combo.split(':')
            light, count = light_count.split('-')
            entries.append({
                'class': class_part,
                'lighting': light,
                'count': int(count)
            })
        
        return {
            'filename': filename,
            'user_id': user_id,
            'sources': entries
        }
    except Exception as e:
        raise ValueError(f"Invalid multi-class filename: {filename}") from e

# ===== Processors =====

def process_same_class_files():
    print(f"\n📂 Parsing same-class filenames from '{SAME_DIR}'\n")
    records = []
    for filename in os.listdir(SAME_DIR):
        if filename.endswith('.csv'):
            try:
                info = parse_same_class_filename(filename)
                print(f"[✓] Parsed: {filename}")
                records.append(info)
            except Exception as e:
                print(f"[!] Skipped {filename}: {e}")
    if SAVE_JSON:
        with open("same_class_parsed.json", 'w') as f:
            json.dump(records, f, indent=2)
        print(f"\n💾 Saved to 'same_class_parsed.json'")

def process_multi_class_files():
    print(f"\n📂 Parsing multi-class filenames from '{MULTI_DIR}'\n")
    records = []
    for filename in os.listdir(MULTI_DIR):
        if filename.endswith('.csv'):
            try:
                info = parse_multi_class_filename(filename)
                print(f"[✓] Parsed: {filename}")
                records.append(info)
            except Exception as e:
                print(f"[!] Skipped {filename}: {e}")
    if SAVE_JSON:
        with open("multi_class_parsed.json", 'w') as f:
            json.dump(records, f, indent=2)
        print(f"\n💾 Saved to 'multi_class_parsed.json'")

# ===== Run =====

if __name__ == "__main__":
    process_same_class_files()
    process_multi_class_files()



📂 Parsing same-class filenames from './artificial_data/combined_same_class'

[✓] Parsed: user02+n+la-6.csv
[✓] Parsed: user02+f+f-13.csv
[✓] Parsed: user02+la+le-10.csv
[✓] Parsed: user02+n+le-6.csv
[✓] Parsed: user02+le+f-6.csv
[✓] Parsed: user02+fl+le-16.csv
[✓] Parsed: user02+fl+f-8.csv
[✓] Parsed: user02+la+f-13.csv
[✓] Parsed: user02+fl+le-13.csv
[✓] Parsed: user02+n+la-16.csv
[✓] Parsed: user02+fl+le-8.csv
[✓] Parsed: user02+le+le-8.csv
[✓] Parsed: user02+fl+fl-13.csv
[✓] Parsed: user02+la+le-8.csv
[✓] Parsed: user02+la+fl-10.csv
[✓] Parsed: user02+fl+f-6.csv
[✓] Parsed: user02+n+n-11.csv
[✓] Parsed: user02+n+f-6.csv
[✓] Parsed: user02+le+la-6.csv
[✓] Parsed: user02+le+fl-8.csv
[✓] Parsed: user02+f+le-11.csv
[✓] Parsed: user02+fl+la-8.csv
[✓] Parsed: user02+fl+f-11.csv
[✓] Parsed: user02+f+la-11.csv
[✓] Parsed: user02+n+n-8.csv
[✓] Parsed: user02+la+n-8.csv
[✓] Parsed: user02+n+la-11.csv
[✓] Parsed: user02+la+la-8.csv
[✓] Parsed: user02+f+le-8.csv
[✓] Parsed: user02+fl+le-11.csv